In [23]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset
import xml.etree.ElementTree as ET
from train_test_helper import get_cell_counts, optimize_split
from sklearn.model_selection import train_test_split
import numpy as np

class BCCDDataset(Dataset):
    def __init__(self, image_dir, anno_dir, indices, transforms=None):
        """
        image_dir : folder containing the .jpg files
        anno_dir  : folder containing the .xml files
        indices   : list of integer indices for this split
                    (these are positions into the full sorted list of images)
        transforms: optional transforms
        """
        
        self.image_dir = image_dir
        self.anno_dir = anno_dir
        self.indices = list(indices)          # make sure it is a list
        self.transforms = transforms
        
        # Class name → integer label
        # 0 is reserved for background by PyTorch detection models
        self.class_to_label = {
            "RBC": 1,
            "WBC": 2,
            "Platelets": 3
        }
        
        # Get a sorted list of all image file names so that
        # integer indices always point to the same image
        self.all_image_files = sorted([
            f for f in os.listdir(image_dir) if f.endswith(".jpg")
        ])

jpg_dir = '../data/BCCD/JPEGImages/'
anno_dir = '../data/BCCD/Annotations/'
xml_data, rbc_counts, wbc_counts, platelet_counts = get_cell_counts(anno_dir)
train_idx, test_idx = train_test_split(np.arange(len(xml_data)), test_size=0.3, random_state=42)
train_idx, test_idx = optimize_split(rbc_counts, wbc_counts, platelet_counts, train_idx, test_idx,verbose=True, only_idx = True)
test_idx, val_idx = train_test_split(test_idx, test_size=0.5, random_state=42)
test_idx_f, val_idx = optimize_split(rbc_counts, wbc_counts, platelet_counts, test_idx, val_idx,verbose=True, only_idx = True)
bc_data = BCCDDataset(jpg_dir, anno_dir, train_idx)


Initial train loss: 0.3417
Initial test loss: 0.7890
Initial Combined Loss: 1.1307
START


100%|██████████| 9000/9000 [00:01<00:00, 6731.59it/s]



Final Train Loss: 0.0434
Final Test Loss: 0.1003
Final Combined Loss: 0.1437

Initial train loss: 0.5153
Initial test loss: 0.5148
Initial Combined Loss: 1.0301
START


100%|██████████| 9000/9000 [00:00<00:00, 9826.48it/s]


Final Train Loss: 0.2083
Final Test Loss: 0.2208
Final Combined Loss: 0.4291
